In [1]:
{
 "cells": [
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "# Medical Search Pattern Analysis\n",
    "Analysis of user search behavior, query patterns, and search effectiveness"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "source": [
    "import sys\n",
    "from pathlib import Path\n",
    "import os\n",
    "from datetime import datetime, timezone\n",
    "import requests\n",
    "import pandas as pd\n",
    "import numpy as np\n",
    "import matplotlib.pyplot as plt\n",
    "import seaborn as sns\n",
    "from sqlalchemy import create_engine\n",
    "from dotenv import load_dotenv\n",
    "\n",
    "%matplotlib inline\n",
    "plt.style.use('seaborn')\n",
    "\n",
    "# Add project root to path\n",
    "project_root = Path.cwd().parent\n",
    "sys.path.append(str(project_root))"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "source": [
    "# SSL Certificate Setup\n",
    "def setup_ssl():\n",
    "    \"\"\"Setup SSL certificate for database connection\"\"\"\n",
    "    ssl_dir = project_root / 'ssl'\n",
    "    ssl_dir.mkdir(exist_ok=True)\n",
    "    cert_path = ssl_dir / 'rds-ca-bundle.pem'\n",
    "    timestamp_path = ssl_dir / 'cert_timestamp.txt'\n",
    "    \n",
    "    current_time = datetime.now(timezone.utc)\n",
    "    \n",
    "    needs_update = True\n",
    "    if cert_path.exists() and timestamp_path.exists():\n",
    "        with open(timestamp_path, 'r') as f:\n",
    "            last_update = datetime.fromisoformat(f.read().strip())\n",
    "            if (current_time - last_update).days < 30:\n",
    "                needs_update = False\n",
    "    \n",
    "    if needs_update:\n",
    "        print(f\"Downloading SSL certificate at {current_time.isoformat()}\")\n",
    "        response = requests.get('https://truststore.pki.rds.amazonaws.com/global/global-bundle.pem')\n",
    "        response.raise_for_status()\n",
    "        \n",
    "        with open(cert_path, 'wb') as f:\n",
    "            f.write(response.content)\n",
    "        \n",
    "        with open(timestamp_path, 'w') as f:\n",
    "            f.write(current_time.isoformat())\n",
    "    \n",
    "    return str(cert_path)\n",
    "\n",
    "SSL_CERT_PATH = setup_ssl()"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "source": [
    "# Database Connection\n",
    "load_dotenv()\n",
    "\n",
    "db_params = {\n",
    "    'host': os.getenv('DB_HOST'),\n",
    "    'database': os.getenv('DB_NAME'),\n",
    "    'user': os.getenv('DB_USER'),\n",
    "    'password': os.getenv('DB_PASSWORD'),\n",
    "    'port': os.getenv('DB_PORT', '5432'),\n",
    "    'sslmode': 'verify-full',\n",
    "    'sslcert': SSL_CERT_PATH\n",
    "}\n",
    "\n",
    "db_url = f\"postgresql://{db_params['user']}:{db_params['password']}@{db_params['host']}:{db_params['port']}/{db_params['database']}\"\n",
    "engine = create_engine(db_url)"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## Query Pattern Analysis"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "source": [
    "# Analyze search patterns over time\n",
    "query = \"\"\"\n",
    "SELECT \n",
    "    DATE_TRUNC('day', TO_TIMESTAMP(log_id)) as search_date,\n",
    "    search_type,\n",
    "    COUNT(*) as search_count,\n",
    "    AVG(results_count) as avg_results,\n",
    "    AVG(CASE WHEN results_count > 0 THEN 1 ELSE 0 END) as success_rate\n",
    "FROM search_logs\n",
    "GROUP BY DATE_TRUNC('day', TO_TIMESTAMP(log_id)), search_type\n",
    "ORDER BY search_date DESC\n",
    "LIMIT 100\n",
    "\"\"\"\n",
    "\n",
    "search_trends = pd.read_sql(query, engine)\n",
    "\n",
    "# Plot trends\n",
    "plt.figure(figsize=(15, 6))\n",
    "for search_type in search_trends['search_type'].unique():\n",
    "    data = search_trends[search_trends['search_type'] == search_type]\n",
    "    plt.plot(data['search_date'], data['search_count'], label=search_type)\n",
    "\n",
    "plt.title('Search Patterns Over Time')\n",
    "plt.xlabel('Date')\n",
    "plt.ylabel('Number of Searches')\n",
    "plt.legend()\n",
    "plt.xticks(rotation=45)\n",
    "plt.tight_layout()\n",
    "plt.show()"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## Query Expansion Analysis"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "source": [
    "# Analyze query expansion effectiveness\n",
    "query = \"\"\"\n",
    "WITH expanded_queries AS (\n",
    "    SELECT \n",
    "        qe.original_query,\n",
    "        qe.expanded_terms,\n",
    "        qe.terms_added,\n",
    "        sl.results_count\n",
    "    FROM query_expansion_logs qe\n",
    "    JOIN search_logs sl ON qe.original_query = sl.query\n",
    ")\n",
    "SELECT \n",
    "    terms_added,\n",
    "    COUNT(*) as query_count,\n",
    "    AVG(results_count) as avg_results\n",
    "FROM expanded_queries\n",
    "GROUP BY terms_added\n",
    "ORDER BY terms_added\n",
    "\"\"\"\n",
    "\n",
    "expansion_metrics = pd.read_sql(query, engine)\n",
    "\n",
    "# Plot expansion effectiveness\n",
    "fig, ax1 = plt.subplots(figsize=(12, 6))\n",
    "\n",
    "ax1.bar(expansion_metrics['terms_added'], expansion_metrics['query_count'], alpha=0.3, color='blue', label='Query Count')\n",
    "ax1.set_xlabel('Number of Terms Added')\n",
    "ax1.set_ylabel('Number of Queries', color='blue')\n",
    "\n",
    "ax2 = ax1.twinx()\n",
    "ax2.plot(expansion_metrics['terms_added'], expansion_metrics['avg_results'], color='red', label='Average Results')\n",
    "ax2.set_ylabel('Average Results', color='red')\n",
    "\n",
    "plt.title('Query Expansion Effectiveness')\n",
    "plt.tight_layout()\n",
    "plt.show()"
   ]
  }
 ],
 "metadata": {
  "kernelspec": {
   "display_name": "Python 3",
   "language": "python",
   "name": "python3"
  }
 }
}

NameError: name 'null' is not defined